In [9]:
# =========================
# STEP 1: IMPORTS
# =========================
import os
import pandas as pd
import numpy as np

# =========================
# STEP 2: LOAD RAW DATA
# =========================
dvc_path = "../data/dvc_raw/raw_data.txt"

if not os.path.exists(dvc_path):
    raise FileNotFoundError(f"File not found at {dvc_path}. Make sure DVC-tracked raw data is present.")

df = pd.read_csv(dvc_path, sep="|", low_memory=False)

# =========================
# STEP 3: CLEAN & PREPROCESS
# =========================
df.columns = df.columns.str.strip().str.replace(" ", "_").str.lower()
df['transactionmonth'] = pd.to_datetime(df['transactionmonth'], errors='coerce', format='%Y-%m-%d %H:%M:%S')
df['vehicleintrodate'] = pd.to_datetime(df['vehicleintrodate'], errors='coerce', format='%m/%Y')

# Fill missing numeric values
df['totalpremium'] = df['totalpremium'].fillna(0)
df['totalclaims'] = df['totalclaims'].fillna(0)
df['capitaloutstanding'] = pd.to_numeric(df['capitaloutstanding'], errors='coerce').fillna(0)
df['customvalueestimate'] = df['customvalueestimate'].fillna(0)
df['newvehicle'] = pd.to_numeric(df['newvehicle'], errors='coerce').fillna(0)

# Convert categorical columns
cat_cols = ['legaltype','title','gender','province','postalcode']
for col in cat_cols:
    df[col] = df[col].astype('category')

# =========================
# STEP 4: DERIVED METRICS
# =========================
df['claim_occurred'] = np.where(df['totalclaims'] > 0, 1, 0)
df['claim_severity'] = df.apply(lambda x: x['totalclaims'] if x['claim_occurred']==1 else np.nan, axis=1)
df['margin'] = df['totalpremium'] - df['totalclaims']

# Save preprocessed data for Task 3
preprocessed_path = "../data/processed/insurance_data_preprocessed.csv"
df.to_csv(preprocessed_path, index=False)
print(f"Preprocessed data saved to {preprocessed_path}")


Preprocessed data saved to ../data/processed/insurance_data_preprocessed.csv
